### Imports and Hyperparameter Settings

In [ ]:
%load_ext autoreload
%autoreload 2

from multimodal_mazes.evolution.genomes.genome_dist import GenomeDist as Genome
from multimodal_mazes.evolution.module_banks.maze_bank import MazeModuleBank
from multimodal_mazes.evolution.module_banks.image_bank import ImageModuleBank

import numpy as np
import unittest
import copy

#### Maze HyperParameters

In [ ]:
HYPERPARAMETERS = {
    'task': 'maze',
    'n_module_types': 3,
    'n_inputs': 8,
    'n_outputs': 4,
    'n_modules': 4,
    'weight_sharing': False,
    'uniform_weights': False, # For weight sharing
    'connectivity': 'UNCONNECTED', # Options: 'FULLY CONNECTED', 'UNCONNECTED', 'SPARSE', 'RANDOM'
    'connection_density' : {'input_density': None, 'output_density': None}, # For 'RANDOM' connectivity
    'population_size': 80,
    'top_genomes': 20, 
    'mutation_rate': 0.8, 
    'crossover_rate': 0.5,
    'one_to_one': True, # Only compatible with UNCONNECTED, SPARSE, and RANDOM if density < 1.0
    'mutation_rates': [0.25, 0.25, 0.25, 0.25],
    'network_type': 1
}

module_bank = MazeModuleBank()

#### Image Hyperparameters

In [ ]:
HYPERPARAMETERS = {
    'task': 'image',
    'n_module_types': 5,
    'n_inputs': 8,
    'n_outputs': 4,
    'n_modules': 4,
    'weight_sharing': False,
    'uniform_weights': False, # For weight sharing
    'connectivity': 'UNCONNECTED', # Options: 'FULLY CONNECTED', 'UNCONNECTED', 'SPARSE', 'RANDOM'
    'connection_density' : {'input_density': None, 'output_density': None}, # For 'RANDOM' connectivity
    'population_size': 80,
    'top_genomes': 20, 
    'mutation_rate': 0.8, 
    'crossover_rate': 0.5,
    'one_to_one': True, # Only compatible with UNCONNECTED, SPARSE, and RANDOM if density < 1.0
    'mutation_rates': [0.25, 0.25, 0.25, 0.25],
    'network_type': 1
}

module_bank = ImageModuleBank(img_type=HYPERPARAMETERS['class_type'])

## Genome

### Unit Testing

In [ ]:
class TestGenome(unittest.TestCase):
    def setUp(self):
        """Set up the test case."""
        self.genome = Genome(genome_id=0, hyperparameters=HYPERPARAMETERS, module_bank=module_bank)

    def test_initialization(self):
        """
        Test the initialization of the Genome class.
        Tests:
            Genome ID is set correctly.
            Modules are initialized correctly.
            Connections are initialized correctly.
        """
        # Check genome properties
        self.assertEqual(self.genome.genome_id, 0)

        # Check module initialization
        self.assertEqual(len(self.genome.mod_grouped_in), HYPERPARAMETERS['n_modules'])
        self.assertEqual(len(self.genome.modules), HYPERPARAMETERS['n_modules'])
        self.assertEqual(len(self.genome.mod_grouped_out), HYPERPARAMETERS['n_outputs'])
        
        # Check connection initialization
        self.assertEqual(self.genome.conn_in_rules, [])
        self.assertEqual(self.genome.conn_out_rules, [])
        
    def test_forward_pass(self):
        """
        Test the forward pass of the Genome class.
        Tests:
            Output is a numpy array.
            Output shape is correct.
        """
        input_vec = np.ones(self.genome.n_inputs)
        output_vec = self.genome.forward_pass(input_vec)

        # Check output properties
        self.assertIsInstance(output_vec, np.ndarray)
        self.assertEqual(output_vec.shape[0], self.genome.n_outputs)

    def test_crossover(self):
        """
        Test the crossover of the Genome class.
        Tests:
            - Crossover produces a new Genome instance.
            - Child has correct genome ID.
            - Child inherits connections from both parents.
        """
        parent_1 = Genome(genome_id=1,hyperparameters=HYPERPARAMETERS)
        parent_2 = Genome(genome_id=2, hyperparameters=HYPERPARAMETERS)
        child = parent_1.crossover(new_id=3, parent_2=parent_2)

        # Check child properties
        self.assertIsInstance(child, Genome)
        self.assertEqual(child.genome_id, 3)

        # Check inheritance from parents
        if len(parent_1.conn_in_rules) > 0 and len(parent_2.conn_in_rules) > 0:
            self.assertNotEqual(child.conn_in_rules, parent_1.conn_in_rules)
            self.assertNotEqual(child.conn_out_rules, parent_2.conn_out_rules)

    def test_mutation(self):
        """
        Test the mutation of the Genome class.
        Tests:
            Mutation modifies the genome's structure.
        """
        mut_genome = Genome(genome_id=5, hyperparameters=HYPERPARAMETERS)
        org_in_rules = copy.deepcopy(mut_genome.conn_in_rules)
        org_out_rules = copy.deepcopy(mut_genome.conn_out_rules)
        for _ in range(10):
            mut_genome.mutate()

        # Check that mutation effects
        self.assertNotEqual(mut_genome.conn_in_rules, org_in_rules)
        self.assertNotEqual(mut_genome.conn_out_rules, org_out_rules)

    def test_clone(self):
        """
        Test the clone method of the Genome class.
        Tests:
            Clone produces a new Genome instance.
            Clone has correct genome ID.
            Clone inherits connections from the original.
        """
        cloned_genome = self.genome.clone(new_id=4)

        # Check cloned properties
        self.assertIsInstance(cloned_genome, Genome)
        self.assertEqual(cloned_genome.genome_id, 4)
        self.assertEqual(self.genome.genome_id, 0)
        self.assertEqual(cloned_genome.compile_flag, 1)

        # Check inherited connections
        for rule1, rule2 in zip(cloned_genome.conn_in_rules, self.genome.conn_in_rules):
            self.assertTupleEqual(rule1, rule2)
        for rule1, rule2 in zip(cloned_genome.conn_out_rules, self.genome.conn_out_rules):
            self.assertTupleEqual(rule1, rule2)

    

In [ ]:
genome_suite = unittest.TestLoader().loadTestsFromTestCase(TestGenome)
unittest.TextTestRunner(verbosity=2).run(genome_suite)

### Output Inspection

#### Initialisation

In [ ]:
genome = Genome(genome_id=0, hyperparameters=HYPERPARAMETERS, module_bank=module_bank)

print("Genome ID:", genome.genome_id)

print("\nInput Connect Rules:")
for rule in genome.conn_in_rules:
    print(rule)

print("\nOutput Connect Rules:")
for rule in genome.conn_out_rules:
    print(rule)

genome.plot_genome()

#### Connectivity

In [ ]:
genome_c = Genome(genome_id=0, hyperparameters=HYPERPARAMETERS, module_bank=module_bank)
genome_c.plot_genome()

#### Crossover

In [ ]:
parent_1 = Genome(genome_id=1, hyperparameters=HYPERPARAMETERS, module_bank=module_bank)
parent_2 = Genome(genome_id=2, hyperparameters=HYPERPARAMETERS, module_bank=module_bank)
child = parent_1.crossover(new_id=3, parent_2=parent_2)

print("Parent 1 Input Connection Rules:"  )
for rule in parent_1.conn_in_rules:
    print(rule)

print("\nParent 1 Output Connection Rules:")
for rule in parent_1.conn_out_rules:
    print(rule)

print("\nParent 2 Input Connection Rules:")
for rule in parent_2.conn_in_rules:
    print(rule)

print("\nParent 2 Output Connection Rules:")
for rule in parent_2.conn_out_rules:
    print(rule)

print("\nChild Input Connection Rules:")
for rule in child.conn_in_rules:
    print(rule)

print("\nChild Output Connection Rules:")
for rule in child.conn_out_rules:
    print(rule)

In [ ]:
parent_1.plot_genome()
parent_2.plot_genome()
child.plot_genome()

#### Mutation

In [ ]:
genome_mut = Genome(genome_id=0, hyperparameters=HYPERPARAMETERS, module_bank=module_bank)

print("Original Input CConnection Rules:")
for rule in genome_mut.conn_in_rules:
    print(rule)

print("\nOriginal Output Connection Rules:")
for rule in genome_mut.conn_out_rules:
    print(rule)

# genome_mut.plot_genome()
for i in range(10):
    genome_mut.mutate()
# genome_mut.plot_genome()

print("\nMutated Input Connection Rules:")
for rule in genome_mut.conn_in_rules:
    print(rule)

print("\nMutated Output Connection Rules:")
for rule in genome_mut.conn_out_rules:
    print(rule)

#### Forward Pass

In [ ]:
genome_fp = Genome(genome_id=0, hyperparameters=HYPERPARAMETERS, module_bank=module_bank)
genome_fp.plot_genome()

In [ ]:
input_vec = np.array([1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0])
print("Input Vector:\n", input_vec)

output_vec = genome_fp.forward_pass(input_vec)
print("Output Vector:\n", output_vec)

In [ ]:
input_vec = np.array([
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
], dtype=float)
input_vec = input_vec.flatten()
print("Input Vector:\n", input_vec)

output_vec = genome_fp.forward_pass(input_vec)
print("Output Vector:\n", output_vec)

#### Compile

In [ ]:
genome_c = Genome(genome_id=0, hyperparameters=HYPERPARAMETERS)
genome_c.plot_genome()

In [ ]:
genome_c.compile_flag = 1
genome_c.compile_rules()

print("Input connectivity rules:")
print(genome_c.mod_grouped_in)
print("Output connectivity rules:")
print(genome_c.mod_grouped_out)